# Re-train Exp C (NaN Fix) + Multi-Seed + Per-Class AP Chart

## Thứ tự chạy (BẮT BUỘC theo thứ tự):
1. **Cell 1** — Install & GPU check
2. **Cell 2** — Clone repo từ GitHub → lấy `src/`, `configs/`
3. **Cell 3** — Mount COCO 2017 & tạo subset
4. **Cell 4** — Setup paths & verify NaN fixes
5. **Cell 5** — Verify ASL implementation
6. **Cell 6** — Re-train Exp C (ưu tiên nhất)
7. **Cell 7** — Multi-seed Exp B (nếu còn giờ)
8. **Cell 8** — Multi-seed Exp A & D
9. **Cell 9** — Per-class AP bar chart
10. **Cell 10** — Ablation summary table

## Cell 1: Install & GPU Check

In [ ]:
!pip install timm pycocotools torchmetrics scikit-learn matplotlib seaborn pyyaml tqdm -q
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'PyTorch: {torch.__version__}')

## Cell 2: Clone Repository từ GitHub

**Đổi `REPO_URL`** thành URL repo thực trên GitHub trước khi chạy.
Sau khi clone, code sẽ nằm tại `/kaggle/working/ECAAL/`.

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/Thinh59/ECAAL.git'  # ← đổi nếu URL khác
REPO_DIR = Path('/kaggle/working/ECAAL')

if REPO_DIR.exists():
    print(f'Repo đã tồn tại — pulling latest...')
    os.system(f'git -C {REPO_DIR} pull')
else:
    print(f'Cloning {REPO_URL} ...')
    ret = os.system(f'git clone {REPO_URL} {REPO_DIR}')
    if ret != 0:
        raise RuntimeError('git clone thất bại! Kiểm tra REPO_URL.')

# Verify
required = [
    'src/train.py', 'src/losses.py', 'src/models.py',
    'src/dataset.py', 'src/evaluate.py', 'src/cbam.py', 'src/utils.py',
    'configs/exp_A_resnet_bce.yaml', 'configs/exp_B_resnet_asl.yaml',
    'configs/exp_C_efficientnet_cbam_asl.yaml', 'configs/exp_D_resnet_focal.yaml',
]
all_ok = True
for rel in required:
    p = REPO_DIR / rel
    ok = p.exists()
    print(f"  {'OK' if ok else 'MISSING'} {rel}")
    if not ok:
        all_ok = False

if not all_ok:
    raise FileNotFoundError('Một số file thiếu — clone chưa đầy đủ!')
print('\nRepository OK!')

## Cell 3: Mount COCO 2017 & Tạo Subset

**Trước khi chạy:** Notebook → **+ Add Data** → search `coco-2017-dataset` (awsaf49) → Add.
Dataset sẽ mount tại `/kaggle/input/coco-2017-dataset/`.

> Nếu `subset_train_ids.json` đã tồn tại từ run trước, cell sẽ **skip** tạo lại.

In [ ]:
import json, sys
from pathlib import Path

COCO_ROOT  = Path('/kaggle/input/coco-2017-dataset')
SUBSET_DIR = Path('/kaggle/working/data/coco_subset')
SUBSET_DIR.mkdir(parents=True, exist_ok=True)

# Verify COCO mount
train_ann = COCO_ROOT / 'annotations' / 'instances_train2017.json'
val_ann   = COCO_ROOT / 'annotations' / 'instances_val2017.json'
for name, p in [('train annotations', train_ann), ('val annotations', val_ann),
                ('train2017/', COCO_ROOT/'train2017'), ('val2017/', COCO_ROOT/'val2017')]:
    print(f"  {'OK' if p.exists() else 'MISSING'} {name}")

if not train_ann.exists():
    raise FileNotFoundError(
        'COCO chưa mount!\n'
        '→ Notebook → + Add Data → coco-2017-dataset (awsaf49)'
    )

# Tạo subset (skip nếu đã có)
train_f = SUBSET_DIR / 'subset_train_ids.json'
val_f   = SUBSET_DIR / 'subset_val_ids.json'

if train_f.exists() and val_f.exists():
    t = json.load(open(train_f)); v = json.load(open(val_f))
    print(f'\nSubset đã có: train={len(t):,}, val={len(v):,} → skip.')
else:
    print('\nTạo subset 16k+4k...')
    sys.path.insert(0, str(REPO_DIR / 'src'))
    from dataset import create_coco_subset
    create_coco_subset(str(COCO_ROOT), str(SUBSET_DIR),
                       num_train=16000, num_val=4000, seed=42)
    print('Subset created!')

## Cell 4: Setup Paths & Verify NaN Fixes

In [ ]:
import os, sys, yaml, json
from pathlib import Path

REPO_DIR   = Path('/kaggle/working/ECAAL')
SRC        = REPO_DIR / 'src'
CONFIGS    = REPO_DIR / 'configs'
OUTPUT_DIR = Path('/kaggle/working/outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(SRC))

cfg_c_path = CONFIGS / 'exp_C_efficientnet_cbam_asl.yaml'
with open(cfg_c_path) as f:
    cfg_c = yaml.safe_load(f)

print('Exp C Config Verification')
checks = [
    ('lr',       cfg_c['optimizer']['lr'],   lambda v: v <= 1e-4, '<= 1e-4'),
    ('eps',      cfg_c['loss'].get('eps',0), lambda v: v >= 1e-5, '>= 1e-5'),
    ('max_norm', cfg_c.get('max_norm',1.0),  lambda v: v <= 0.5,  '<= 0.5'),
]
all_ok = True
for name, val, cond, desc in checks:
    ok = cond(val)
    print(f"  {'OK' if ok else 'FAIL'} {name}={val} (expected {desc})")
    if not ok: all_ok = False

if not all_ok:
    raise RuntimeError('Config fixes chưa đúng — kiểm tra exp_C_efficientnet_cbam_asl.yaml')
print('\nConfig OK — sẵn sàng re-train!')

## Cell 5: Verify ASL (không có NaN trên logits bão hoà)

In [ ]:
import torch, inspect
from losses import AsymmetricLoss

src = inspect.getsource(AsymmetricLoss.forward)
correct = 'xs_pos_shifted = (xs_pos - self.clip).clamp(min=0)' in src
bug     = 'xs_neg_shifted = (xs_neg + self.clip)' in src
print(f"  {'OK' if correct else 'FAIL'} correct shift")
print(f"  {'OK' if not bug else 'FAIL'} old bug absent")

asl = AsymmetricLoss(gamma_pos=0, gamma_neg=4, clip=0.05, eps=1e-5)
logits  = torch.tensor([[10., -10., 5., -5.]] * 8)
targets = torch.randint(0, 2, (8, 4)).float()
loss = asl(logits, targets)
nan_ok = not torch.isnan(loss)
print(f"  {'OK' if nan_ok else 'FAIL'} no NaN on saturated logits (loss={loss.item():.4f})")

if correct and not bug and nan_ok:
    print('\nASL OK')
else:
    raise RuntimeError('ASL có vấn đề — kiểm tra src/losses.py')

## Cell 6: Re-train Exp C (EfficientNet-B0 + CBAM + ASL — NaN Fixed)

Fixes so với run bị NaN: `lr 3e-4→1e-4`, `eps 1e-8→1e-5`, `max_norm 1.0→0.5`

Kỳ vọng: không còn NaN từ epoch 12, mAP ~0.77–0.78.

In [ ]:
import time
from train import run

t0 = time.time()
print(f'Config: {cfg_c_path}')
print(f"lr={cfg_c['optimizer']['lr']}, eps={cfg_c['loss'].get('eps')}, max_norm={cfg_c.get('max_norm')}")

best_c_v2 = run(str(cfg_c_path))
elapsed = (time.time() - t0) / 3600

print(f'\n[Done] Exp C (v2) | mAP={best_c_v2:.4f} | {elapsed:.1f}h')
print(f'  Exp C old (NaN): 0.7516')
print(f'  Exp C new (fix): {best_c_v2:.4f}')
print(f'  Exp B (target) : 0.7913')

## Cell 7: Multi-Seed Exp B (ResNet50 + ASL)

Chạy seeds 42, 123, 2025 → báo cáo `mean ± std` cho paper (KSE requirement).

In [ ]:
import yaml, copy, numpy as np
from train import run

SEEDS = [42, 123, 2025]
cfg_b_path = CONFIGS / 'exp_B_resnet_asl.yaml'
with open(cfg_b_path) as f:
    cfg_b_base = yaml.safe_load(f)

maps_b = {}
for seed in SEEDS:
    cfg_tmp = copy.deepcopy(cfg_b_base)
    cfg_tmp['seed'] = seed
    cfg_tmp['output_dir'] = f'/kaggle/working/outputs_multiseed/expB_seed{seed}'
    tmp = f'/kaggle/working/tmp_B_s{seed}.yaml'
    with open(tmp, 'w') as f:
        yaml.dump(cfg_tmp, f)
    print(f'\n--- Exp B | Seed {seed} ---')
    maps_b[seed] = run(tmp)
    print(f'  mAP = {maps_b[seed]:.4f}')

vals = list(maps_b.values())
print(f'\nExp B Multi-Seed: mean={np.mean(vals):.4f} ± std={np.std(vals):.4f}')

## Cell 8: Multi-Seed Exp A (BCE) & Exp D (Focal)

In [ ]:
import yaml, copy, json, numpy as np
from train import run

SEEDS = [42, 123, 2025]
multiseed = {}

for exp_tag, cfg_file in [
    ('A', CONFIGS / 'exp_A_resnet_bce.yaml'),
    ('D', CONFIGS / 'exp_D_resnet_focal.yaml'),
]:
    with open(cfg_file) as f:
        base = yaml.safe_load(f)
    seed_maps = {}
    for seed in SEEDS:
        cfg_tmp = copy.deepcopy(base)
        cfg_tmp['seed'] = seed
        cfg_tmp['output_dir'] = f'/kaggle/working/outputs_multiseed/exp{exp_tag}_seed{seed}'
        tmp = f'/kaggle/working/tmp_{exp_tag}_s{seed}.yaml'
        with open(tmp, 'w') as f:
            yaml.dump(cfg_tmp, f)
        print(f'Exp {exp_tag} | Seed {seed}')
        seed_maps[seed] = run(tmp)
        print(f'  mAP = {seed_maps[seed]:.4f}')
    vals = list(seed_maps.values())
    multiseed[f'Exp_{exp_tag}'] = {'mean': float(np.mean(vals)), 'std': float(np.std(vals)), 'per_seed': seed_maps}

# Add Exp B if available
if 'maps_b' in dir():
    vals = list(maps_b.values())
    multiseed['Exp_B'] = {'mean': float(np.mean(vals)), 'std': float(np.std(vals)), 'per_seed': maps_b}

print('\nMULTI-SEED SUMMARY')
for k, v in multiseed.items():
    print(f'  {k}: {v["mean"]:.4f} ± {v["std"]:.4f}')

with open('/kaggle/working/multiseed_results.json', 'w') as f:
    json.dump(multiseed, f, indent=2)
print('\nSaved: /kaggle/working/multiseed_results.json')

## Cell 9: Per-Class AP Bar Chart (Figure 2 — Paper)

So sánh AP từng class giữa Exp A (BCE) và Exp B (ASL). Highlight rare classes.

In [ ]:
import json, os, numpy as np, matplotlib.pyplot as plt

BASE = '/kaggle/working/outputs'

def load_best(path):
    recs = json.load(open(path))
    return max(recs, key=lambda r: r.get('mAP', 0))

best_a = load_best(f'{BASE}/exp_A_resnet_bce/log.json')
best_b = load_best(f'{BASE}/exp_B_resnet_asl/log.json')
ap_a, ap_b = np.array(best_a['AP_per_class']), np.array(best_b['AP_per_class'])
delta = ap_b - ap_a

COCO_CLASSES = [
    'person','bicycle','car','motorcycle','airplane','bus','train','truck',
    'boat','traffic light','fire hydrant','stop sign','parking meter','bench',
    'bird','cat','dog','horse','sheep','cow','elephant','bear','zebra','giraffe',
    'backpack','umbrella','handbag','tie','suitcase','frisbee','skis','snowboard',
    'sports ball','kite','baseball bat','baseball glove','skateboard','surfboard',
    'tennis racket','bottle','wine glass','cup','fork','knife','spoon','bowl',
    'banana','apple','sandwich','orange','broccoli','carrot','hot dog','pizza',
    'donut','cake','chair','couch','potted plant','bed','dining table','toilet',
    'tv','laptop','mouse','remote','keyboard','cell phone','microwave','oven',
    'toaster','sink','refrigerator','book','clock','vase','scissors','teddy bear',
    'hair drier','toothbrush'
]
RARE = {'scissors','toaster','hair drier'}

idx_top = np.argsort(delta)[::-1][:30]
x, w = np.arange(30), 0.35

fig, ax = plt.subplots(figsize=(14, 7))
ax.bar(x - w/2, ap_a[idx_top], w, label=f'Exp A BCE (mAP={best_a["mAP"]:.4f})', color='#e74c3c', alpha=0.85)
ax.bar(x + w/2, ap_b[idx_top], w, label=f'Exp B ASL (mAP={best_b["mAP"]:.4f})', color='#3498db', alpha=0.85)

labels = [COCO_CLASSES[i] for i in idx_top]
for i, cls_idx in enumerate(idx_top):
    if COCO_CLASSES[cls_idx] in RARE:
        ax.annotate('*', xy=(i, max(ap_a[cls_idx], ap_b[cls_idx]) + 0.03),
                    ha='center', fontsize=16, color='#f39c12', fontweight='bold')
    d = delta[cls_idx]
    ax.text(i+w/2, ap_b[cls_idx]+0.01, f'+{d:.2f}' if d>=0 else f'{d:.2f}',
            ha='center', fontsize=6, color='#27ae60' if d>=0 else '#e74c3c', fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Average Precision (AP)')
ax.set_title('Top 30 Classes by ΔMAP: Exp A (BCE) vs Exp B (ASL)  (* = rare class)', fontweight='bold')
ax.set_ylim(0, 1.15)
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
out = f'{BASE}/per_class_ap_chart.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Chart saved: {out}')

## Cell 10: Ablation Summary Table (Export for Paper)

In [ ]:
import json, os, pandas as pd

BASE = '/kaggle/working/outputs'
exps = [
    ('A','ResNet50','BCE',False,'exp_A_resnet_bce'),
    ('D','ResNet50','Focal',False,'exp_D_resnet_focal'),
    ('B','ResNet50','ASL (fixed)',False,'exp_B_resnet_asl'),
    ('C','EfficientNet-B0','ASL (fixed)',True,'exp_C_efficientnet_cbam_asl'),
]
rows = []
base_map = None
for exp_id, backbone, loss, cbam, folder in exps:
    p = f'{BASE}/{folder}/log.json'
    if not os.path.exists(p):
        print(f'Missing: {p}'); continue
    recs = json.load(open(p))
    best = max(recs, key=lambda r: r.get('mAP', 0))
    if exp_id == 'A':
        base_map = best['mAP']
    delta = f"+{best['mAP']-base_map:.4f}" if base_map and exp_id != 'A' else '—'
    rows.append({'Exp':exp_id,'Backbone':backbone,'Loss':loss,'CBAM':'Y' if cbam else 'N',
                 'Best Ep':best['epoch'],'mAP':f"{best['mAP']:.4f}",
                 'dMAP':delta,'Macro-F1':f"{best['macro_f1']:.4f}"})

df = pd.DataFrame(rows)
print('\nABLATION STUDY — FINAL')
print(df.to_string(index=False))
out = f'{BASE}/ablation_final_table.csv'
df.to_csv(out, index=False)
print(f'\nSaved: {out}')